In [1]:
import os

base_path = "/content/Self-Improving-Agent-Projects"

folders = [
    "modules/01_test_time_compute",
    "modules/02_robust_verification",
    "shared",
    ".codex"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("✅ Folder structure created")

✅ Folder structure created


# HW: Test-Time Compute Strategies on GSM8K

This notebook reflects the course material in module Stanford's CS239A course. In this notebook, we will explore various test time compute strategies to help improve the results of a small model. The first module illustrates that 'more compute' is typically associated with better performance. Using strategies to increase prompting can help our end result.

We will use the **GSM8K** dataset and a **lightweight open model** (**Qwen/Qwen2.5-1.5B-Instruct**) with 4-bit quantization. We will show how a lightweight model can achieve improved results with test time compute strategies. We will explore the following techniques in this notebook:

- **Part 1:** Single-pass CoT baseline (System-1)
- **Part 2:** Best-of-N sampling + **majority vote** (self-consistency)
- **Part 3:** Best-of-N + **simple verifier** (Proposer–Verifier using labels)
- **Part 4:** **Difficulty binning** (Easy/Medium/Hard) via empirical pass rate
- **Part 5:** **Sequential revisions** (self-refinement chains)
- **Part 6:** Lightweight **beam/search** over partial reasoning
- **Part 7:** Difficulty-aware **routing policy** (compute-optimal TTC)

## Part 0 — Setup
Set up the environment, load GSM8K and load a lightweight model. For this assignment, we will use Qwen2.5-1.5B Instruct. More information can be found about the model at the link below.

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct



In [3]:
!pip -q install -U bitsandbytes transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.4 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

import torch
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
import os
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# Global timer start
NOTEBOOK_START_TIME = time.time()


### Shared Helper Utilities

To keep this notebook readable and modular, reusable utility functions are stored in:

`shared/helpers.py`

This helper module contains common logic used throughout the notebook, including:

- answer extraction
- example display helpers
- single-generation utilities
- majority voting
- evaluation logic for both the baseline and self-consistency settings

In [ ]:
# Load helper utilities from shared module
_repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "helpers.py").exists()),
    None
)

if _repo_root is None:
    raise FileNotFoundError("Could not locate shared/helpers.py from the current notebook directory.")

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import shared.helpers as helpers
import importlib
importlib.reload(helpers)

# Import commonly used functions
final_re = helpers.final_re
extract_true_answer = helpers.extract_true_answer
extract_pred_answer = helpers.extract_pred_answer
show_example = helpers.show_example
generate_one = helpers.generate_one
majority_vote = helpers.majority_vote
evaluate = helpers.evaluate
build_generation_pool = helpers.build_generation_pool
evaluate_from_generation_pool = helpers.evaluate_from_generation_pool
evaluate_best_of_n_simple_verifier = helpers.evaluate_best_of_n_simple_verifier
build_difficulty_table_from_pool = helpers.build_difficulty_table_from_pool
build_revision_prompt = helpers.build_revision_prompt
run_revision_chain = helpers.run_revision_chain
select_revision_attempt = helpers.select_revision_attempt
evaluate_revision_chains = helpers.evaluate_revision_chains
build_step_prompt = helpers.build_step_prompt
generate_continuations_with_scores = helpers.generate_continuations_with_scores
beam_search_reasoning = helpers.beam_search_reasoning
evaluate_beam_search = helpers.evaluate_beam_search
run_routed_strategy = helpers.run_routed_strategy
evaluate_routing_policy = helpers.evaluate_routing_policy
summarize_policy_results = helpers.summarize_policy_results

# Shared experiment configuration
MAX_NEW_TOKENS = 2000
TEMPERATURE = 0.7
N_SAMPLES = 5
MAX_QUESTIONS = 50

BASELINE_N_SAMPLES = 1
REVISION_STEPS = 3
BEAM_SIZE = 3
BEAM_EXPANSIONS = 2
BEAM_MAX_STEPS = 4
BEAM_LENGTH_PENALTY = 0.02
VERIFIER_MODE = "exact"

SOLVER_SYSTEM_PROMPT = """You are an efficient math tutor. Solve the math problem.
Show clear step-by-step reasoning, verify the calculations, and then give one final numeric answer.
Only complete necessary steps"""

REVISION_PROMPT_TEMPLATE = """You previously attempted the following math problem.
Your earlier reasoning may contain mistakes.
Carefully review the previous attempt, correct any errors, and produce an improved solution.
End your response with a final numeric answer. Only complete necessary steps.

Question:
{question}

Previous attempt:
{previous_attempt_text}
"""

print("✅ Helper utilities and configurations loaded")


In [ ]:
# Load GSM8K dataset
dataset = load_dataset("gsm8k", "main")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(f"GSM8K train size: {len(train_df)}")
print(f"GSM8K test size: {len(test_df)}")
print("\nFirst few examples:")
train_df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully!
Training subset: 1000 examples
Evaluation subset: 500 examples


In [ ]:
# Inspect training examples
print("GSM8K answer format example:\n")
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 

## Answer Pattern and Regex

It is important to understand the structure of a data set To appropriately prompt a reasoning model. GSM8K solutions end with a standardized final-answer line:

- Final numeric answer appears after `####`
- It is on its own line
- It may be an integer or decimal


Because this format is consistent, we can create a regex pattern to reliably extract the true answer with a regular expression. The regex pattern below is defined in shared/helpers.py:

`final_re = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")
`


In [ ]:
# Display answer extraction regex pattern
print("Final answer regex pattern:", final_re.pattern)
print("\nFormat: Extracts numeric value after '####' marker")

# Verify extraction works correctly
dataset = dataset.map(lambda x: {"true_answer": extract_true_answer(x)})

print("\n✅ Answers extracted and verified")


Final answer regex: ####\s*([-+]?\d+(?:\.\d+)?)


## Model Setup

In this section, we load a lightweight instruction-tuned language model:

- **Model:** Qwen2.5-1.5B-Instruct  
- **Type:** Causal Language Model (decoder-only transformer)  
- **Use case:** Step-by-step reasoning and instruction following  

This model is intentionally small (~1.5B parameters) so it can run efficiently in a Colab environment while still demonstrating reasoning capabilities. We will use Quantitization which significantly reduces GPU usage without a severe cost to precision.  We will use **4-bit quantization** via the `bitsandbytes` library.


In [ ]:
# Check GPU and load 4-bit quantized model
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))

model_name = "Qwen/Qwen3-1.7B"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.eval()
print(f"✅ Model loaded: {model_name}")


CUDA Available: True
GPU: Tesla T4
Compute Capability: (7, 5)


In [ ]:
# Create evaluation subsets for faster testing
train_subset = dataset["train"].select(range(1000))
eval_subset = dataset["test"].select(range(500))
eval_df = pd.DataFrame(eval_subset)

print(f"Evaluation subset: {len(eval_df)} examples")

# Prepare dataset - create subsets
train_subset_small = dataset["train"].select(range(1000))
eval_subset_small = dataset["test"].select(range(500))

print("✅ Dataset loaded and prepared")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-1.7B


## View The Output of A Few Sample Model Generations

It is important to ensure that the output of the model is as expected. It is important to take note of factors such as the maximum output token length, and the quality of the prompt provided to the model.

1. MAX_NEW_TOKENS Is set to 2000. if the model S output exceeds 2000 tokens, the final answer will not be extracted.

2. The model is asked to perform COT reasoning (ie. show 'step by step' how to solve the problem).

3. The model is instructed to only complete necessary steps. This helps keep the model output within the token limit.

In [ ]:
# Test model generations on a few examples
print("Sample model outputs:\n")
print("=" * 80)

new_token_counts = []

for i in range(5):
    print(f"\n--- Example {i+1} ---")
    q = eval_df.iloc[i]["question"]
    true = extract_true_answer(eval_df.iloc[i]["answer"])

    text, pred, in_tok, new_tok, total_tok = generate_one(
        question=q,
        model=model,
        tokenizer=tokenizer,
        temperature=TEMPERATURE,
        max_new_tokens=MAX_NEW_TOKENS,
        system_prompt=SOLVER_SYSTEM_PROMPT,
    )

    new_token_counts.append(new_tok)

    print(f"Q: {q[:100]}...")
    print(f"Predicted: {pred}, True: {true}")
    print(f"Tokens - Input: {in_tok}, Output: {new_tok}, Total: {total_tok}")

print("\n" + "=" * 80)
print(f"Token statistics:")
print(f"  Avg output tokens: {np.mean(new_token_counts):.2f}")
print(f"  Min output tokens: {np.min(new_token_counts)}")
print(f"  Max output tokens: {np.max(new_token_counts)}")


QUESTION:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL OUTPUT:
 system
You are an efficient math tutor. Solve the math problem. 
Show clear step-by-step reasoning, verify the calculations, and then give one final numeric answer. 
Only complete necessary steps
user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
assistant
<think>
Okay, let's try to figure out how much money Janet makes every day at the farmers' market. Hmm, let's start by breaking down the information given. 

First, Janet's ducks lay 16 eggs per day. 

## Part 1 — Baseline (Single-Pass Chain-of-Thought)

The baseline generates one CoT solution per question without any sampling or reranking. This establishes a performance floor and lets us measure the value of test-time compute strategies.


In [ ]:
# Part 1: Baseline Evaluation
baseline_metrics, baseline_results = evaluate(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=BASELINE_N_SAMPLES,
    temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt=SOLVER_SYSTEM_PROMPT,
)

print("=" * 80)
print("PART 1: BASELINE RESULTS")
print("=" * 80)
print(json.dumps(baseline_metrics, indent=2))


Eval: baseline:   6%|▌         | 3/50 [06:58<1:53:41, 145.14s/it]

## Parts 2 & 3 — Best-of-N Methods (Majority Vote & Verifier)

We generate **the same set of N CoT samples per question** and compare two selection methods:

- **Part 2: Majority Vote** — Select the most common final answer  
- **Part 3: Verifier-Based** — Score each sample and select the best one  

Using a shared sample pool makes the comparison clean: both methods see the same candidates, so differences come purely from the **selection strategy**, not sampling variance.


In [ ]:
# Build one shared answer pool for Parts 2 and 3
answer_pool, answer_pool_meta = build_generation_pool(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=N_SAMPLES,
    temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt=SOLVER_SYSTEM_PROMPT,
)

print("Shared answer pool built.")
print(json.dumps(answer_pool_meta, indent=2))

In [ ]:
import os
import json
from google.colab import files

# Save inside the current Colab runtime
SAVE_DIR = "/content/Self-Improving-Agent-Projects/.codex/artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    SAVE_DIR,
    f"gsm8k_pool_q{MAX_QUESTIONS}_n{N_SAMPLES}_temp{TEMPERATURE}.json"
)

with open(SAVE_PATH, "w") as f:
    json.dump(
        {
            "answer_pool": answer_pool,
            "meta": answer_pool_meta
        },
        f,
        indent=2
    )

print(f"Saved answer pool to: {SAVE_PATH}")

# Trigger browser download to your local machine
files.download(SAVE_PATH)

## Part 2 — Self-Consistency (Majority Vote)

We now generate **multiple independent CoT solutions** per question and use **majority voting** to select the final answer.

**Approach:**
- Generate N samples for each question using different random seeds
- Extract the final numeric answer from each sample
- Select the **most frequent answer** across all samples

**Example:**
- Samples → `[18, 18, 22, 18, 22]`  
- Final answer via majority vote → **18**

This approach (also called **self-consistency**) improves accuracy by leveraging multiple reasoning paths without requiring a learned verifier.


In [ ]:
# Build one shared answer pool for Parts 2 and 3
answer_pool, answer_pool_meta = build_generation_pool(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=N_SAMPLES,
    temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt=SOLVER_SYSTEM_PROMPT,
)

print("✅ Shared answer pool built")
print(json.dumps(answer_pool_meta, indent=2))

# Part 2: Self-Consistency from the shared answer pool
sc_metrics, sc_results = evaluate_from_generation_pool(
    generation_pool=answer_pool,
    pool_meta=answer_pool_meta,
    selection_mode="majority_vote",
)

print("\n" + "=" * 80)
print("Self-Consistency (Majority Vote) Results:")
print(json.dumps(sc_metrics, indent=2))


In [ ]:
# Part 3: Verifier-Based Selection
verifier_metrics, verifier_results = evaluate_from_generation_pool(
    generation_pool=answer_pool,
    pool_meta=answer_pool_meta,
    selection_mode="simple_verifier",
    verifier_mode="exact",
)

print("\n" + "=" * 80)
print("Verifier-Based Selection Results:")
print(json.dumps(verifier_metrics, indent=2))

# Part 4: Difficulty Binning
difficulty_df = build_difficulty_table_from_pool(answer_pool)

print("\n" + "=" * 80)
print("Difficulty Bins (based on empirical pass rate):")
print(difficulty_df["difficulty"].value_counts())
print("\nDifficulty mapping:")
print("  Easy: pass_rate ≥ 0.8")
print("  Medium: 0.3 ≤ pass_rate < 0.8")
print("  Hard: pass_rate < 0.3")


## Part 4 — Difficulty Binning (Easy / Medium / Hard)

We estimate problem difficulty by computing the empirical pass rate across sampled solutions:

- **Easy:** pass rate ≥ 0.8 (model consistently solves)  
- **Medium:** 0.3 ≤ pass rate < 0.8 (mixed results)  
- **Hard:** pass rate < 0.3 (rarely solved)

This allows us to understand which problems are reliably solved vs. unstable, which is useful for Part 7's routing policy.


In [ ]:
# Part 5: Sequential Revisions
# Attach difficulty labels to evaluation set
eval_with_diff = eval_df.iloc[:len(difficulty_df)].copy().reset_index().rename(columns={"index": "idx"})
eval_with_diff = eval_with_diff.merge(
    difficulty_df[["idx", "difficulty", "pass_rate"]],
    on="idx",
    how="left"
)

# Test three revision selection strategies
rev_last_metrics, _ = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=REVISION_STEPS,
    base_temperature=TEMPERATURE,
    rev_temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    base_system_prompt=SOLVER_SYSTEM_PROMPT,
    revision_prompt_template=REVISION_PROMPT_TEMPLATE,
    revision_system_prompt=SOLVER_SYSTEM_PROMPT,  # Using same as base for simplicity
    selection_mode="last",
)

rev_mv_metrics, _ = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=REVISION_STEPS,
    base_temperature=TEMPERATURE,
    rev_temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    base_system_prompt=SOLVER_SYSTEM_PROMPT,
    revision_prompt_template=REVISION_PROMPT_TEMPLATE,
    revision_system_prompt=SOLVER_SYSTEM_PROMPT,
    selection_mode="majority_vote",
)

rev_ver_metrics, _ = evaluate_revision_chains(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    r_steps=REVISION_STEPS,
    base_temperature=TEMPERATURE,
    rev_temperature=TEMPERATURE,
    max_questions=MAX_QUESTIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    base_system_prompt=SOLVER_SYSTEM_PROMPT,
    revision_prompt_template=REVISION_PROMPT_TEMPLATE,
    revision_system_prompt=SOLVER_SYSTEM_PROMPT,
    selection_mode="simple_verifier",
    verifier_mode=VERIFIER_MODE,
)

print("=" * 80)
print("Part 5: Sequential Revisions Results\n")
print("Last revision:"); print(json.dumps(rev_last_metrics, indent=2))
print("\nMajority vote:"); print(json.dumps(rev_mv_metrics, indent=2))
print("\nSimple verifier:"); print(json.dumps(rev_ver_metrics, indent=2))


## Part 6 — Lightweight Beam Search

Unlike Best-of-N (parallel sampling) or Sequential Revisions (depth), beam search keeps multiple partial reasoning paths alive and expands them incrementally:

1. Start with a small beam of partial reasoning states
2. Expand each state with candidate next steps  
3. Rank and keep top-K by log-probability + length penalty
4. Continue until a final answer appears or max steps reached

This provides a more structured search than independent sampling, bridging breadth and depth approaches.


In [ ]:
# Part 6: Beam Search
beam_metrics, _ = evaluate_beam_search(
    eval_df=eval_with_diff,
    model=model,
    tokenizer=tokenizer,
    beam_size=BEAM_SIZE,
    expansions_per_beam=BEAM_EXPANSIONS,
    max_steps=BEAM_MAX_STEPS,
    temperature=TEMPERATURE,
    step_max_new_tokens=MAX_NEW_TOKENS,
    length_penalty=BEAM_LENGTH_PENALTY,
    verifier_mode=VERIFIER_MODE,
    max_questions=MAX_QUESTIONS,
)

print("=" * 80)
print("Part 6: Beam Search Results")
print(json.dumps(beam_metrics, indent=2))


## Part 7 — Difficulty-Aware Routing Policy

Instead of using the same strategy for all problems, we route different problems to different methods based on their empirical difficulty.

**Policies:**
- **Policy A:** Easy → S1, Medium → S2, Hard → S3
- **Policy B:** Easy → S1, Medium → S3, Hard → S4  
- **Policy C:** Easy → S2, Medium → S3, Hard → S4

Where:
- **S1** = Baseline (1 sample)
- **S2** = Self-consistency (5 samples + majority vote)
- **S3** = Revisions (3 revision steps + verifier)
- **S4** = Beam search

This balances accuracy with computational cost by allocating more compute to harder problems.


In [ ]:
policy_A = {"easy": "S1", "medium": "S2", "hard": "S3"}
policy_B = {"easy": "S1", "medium": "S3", "hard": "S4"}
policy_C = {"easy": "S2", "medium": "S3", "hard": "S4"}

cfg_map = {
    "S1": {
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "system_prompt": SOLVER_SYSTEM_PROMPT,
    },
    "S2": {
        "n_samples": N_SAMPLES,
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "system_prompt": SOLVER_SYSTEM_PROMPT,
    },
    "S3": {
        "r_steps": REVISION_STEPS,
        "base_temperature": TEMPERATURE,
        "rev_temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "base_system_prompt": SOLVER_SYSTEM_PROMPT,
        "revision_prompt_template": REVISION_PROMPT_TEMPLATE,
        "revision_system_prompt": REVISION_SYSTEM_PROMPT,
        "selection_mode": "simple_verifier",
        "verifier_mode": VERIFIER_MODE,
    },
    "S4": {
        "beam_size": BEAM_SIZE,
        "expansions_per_beam": BEAM_EXPANSIONS,
        "max_steps": BEAM_MAX_STEPS,
        "temperature": TEMPERATURE,
        "step_max_new_tokens": MAX_NEW_TOKENS,
        "length_penalty": BEAM_LENGTH_PENALTY,
        "verifier_mode": VERIFIER_MODE,
        "step_prompt_template": BEAM_STEP_PROMPT_TEMPLATE,
        "step_system_prompt": BEAM_STEP_SYSTEM_PROMPT,
    },
}

In [ ]:
policy_subset = eval_with_diff.copy()

polA_metrics, polA_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_A,
    cfg_map=cfg_map,
    max_questions=MAX_QUESTIONS,
    policy_name="Policy A",
)

polB_metrics, polB_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_B,
    cfg_map=cfg_map,
    max_questions=MAX_QUESTIONS,
    policy_name="Policy B",
)

polC_metrics, polC_rows = evaluate_routing_policy(
    eval_df=policy_subset,
    model=model,
    tokenizer=tokenizer,
    policy_map=policy_C,
    cfg_map=cfg_map,
    max_questions=MAX_QUESTIONS,
    policy_name="Policy C",
)

print("Policy A:")
print(json.dumps(polA_metrics, indent=2))

print("\nPolicy B:")
print(json.dumps(polB_metrics, indent=2))

print("\nPolicy C:")
print(json.dumps(polC_metrics, indent=2))

In [ ]:
# Add a compact comparison table cell
policy_summary_df = summarize_policy_results({
    "Policy A": polA_metrics,
    "Policy B": polB_metrics,
    "Policy C": polC_metrics,
})

policy_summary_df

### Part 7 Results Summary

Difficulty-aware routing lets us allocate compute more selectively across problems.

Compared with a uniform strategy, a routing policy can sometimes achieve a better tradeoff between:
- overall accuracy
- average token usage
- runtime

This makes Part 7 the most practical section of the notebook, because it turns the earlier TTC methods into a simple decision policy rather than treating every problem the same way.

In [ ]:
# Part 7: Difficulty-Aware Routing Policy
policy_A = {"easy": "S1", "medium": "S2", "hard": "S3"}
policy_B = {"easy": "S1", "medium": "S3", "hard": "S4"}
policy_C = {"easy": "S2", "medium": "S3", "hard": "S4"}

cfg_map = {
    "S1": {"temperature": TEMPERATURE, "max_new_tokens": MAX_NEW_TOKENS, "system_prompt": SOLVER_SYSTEM_PROMPT},
    "S2": {"n_samples": N_SAMPLES, "temperature": TEMPERATURE, "max_new_tokens": MAX_NEW_TOKENS, "system_prompt": SOLVER_SYSTEM_PROMPT},
    "S3": {
        "r_steps": REVISION_STEPS,
        "base_temperature": TEMPERATURE,
        "rev_temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "base_system_prompt": SOLVER_SYSTEM_PROMPT,
        "revision_prompt_template": REVISION_PROMPT_TEMPLATE,
        "revision_system_prompt": SOLVER_SYSTEM_PROMPT,
        "selection_mode": "simple_verifier",
        "verifier_mode": VERIFIER_MODE,
    },
    "S4": {
        "beam_size": BEAM_SIZE,
        "expansions_per_beam": BEAM_EXPANSIONS,
        "max_steps": BEAM_MAX_STEPS,
        "temperature": TEMPERATURE,
        "step_max_new_tokens": MAX_NEW_TOKENS,
        "length_penalty": BEAM_LENGTH_PENALTY,
        "verifier_mode": VERIFIER_MODE,
    },
}

policy_subset = eval_with_diff.copy()

polA_metrics, _ = evaluate_routing_policy(eval_df=policy_subset, model=model, tokenizer=tokenizer, 
                                           policy_map=policy_A, cfg_map=cfg_map, max_questions=MAX_QUESTIONS, policy_name="Policy A")
polB_metrics, _ = evaluate_routing_policy(eval_df=policy_subset, model=model, tokenizer=tokenizer, 
                                           policy_map=policy_B, cfg_map=cfg_map, max_questions=MAX_QUESTIONS, policy_name="Policy B")
polC_metrics, _ = evaluate_routing_policy(eval_df=policy_subset, model=model, tokenizer=tokenizer, 
                                           policy_map=policy_C, cfg_map=cfg_map, max_questions=MAX_QUESTIONS, policy_name="Policy C")

print("=" * 80)
print("Part 7: Routing Policy Results\n")
print("Policy A:", json.dumps(polA_metrics, indent=2))
print("\nPolicy B:", json.dumps(polB_metrics, indent=2))
print("\nPolicy C:", json.dumps(polC_metrics, indent=2))

# Summary comparison
policy_summary_df = summarize_policy_results({
    "Policy A": polA_metrics,
    "Policy B": polB_metrics,
    "Policy C": polC_metrics,
})
print("\nPolicy Comparison:\n", policy_summary_df)


In [ ]:
# Create comprehensive results comparison
metric_rows = []
for m in [baseline_metrics, sc_metrics, verifier_metrics, rev_last_metrics, rev_mv_metrics, rev_ver_metrics]:
    if isinstance(m, dict):
        metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows)

print("All Methods Comparison:")
print(metrics_df.to_string())


In [ ]:
# Visualization: Accuracy vs Test-Time Compute
if "accuracy" in metrics_df.columns and "avg_tokens" in metrics_df.columns:
    plt.figure(figsize=(10, 6))
    plt.scatter(metrics_df["avg_tokens"], metrics_df["accuracy"], s=100, alpha=0.6)
    for _, r in metrics_df.iterrows():
        label = r.get("method", "")
        if label:
            plt.annotate(label, (r["avg_tokens"], r["accuracy"]), fontsize=9, ha='right')
    plt.xlabel("Average Tokens (Test-Time Compute Proxy)")
    plt.ylabel("Accuracy")
    plt.title("Test-Time Compute Efficiency Tradeoff")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Key Takeaways and Next Steps

This notebook demonstrates how **test-time compute can improve model performance** across multiple strategies:

1. **Baseline** establishes the single-pass floor
2. **Majority vote** shows that sampling multiple paths helps when problems have ambiguity
3. **Verifier selection** tests whether learned or heuristic scoring can outperform voting
4. **Difficulty binning** reveals which problems are easy vs hard for the model
5. **Sequential revisions** explores whether the model can self-correct
6. **Beam search** structures the search process over partial reasoning
7. **Routing policies** combine methods wisely to trade accuracy vs compute

**Next Steps:**
- Run with larger evaluation sets (200+ questions) for statistical significance
- Try different verifier designs (learned vs heuristic)
- Experiment with different difficulty thresholds
- Compare wall-clock runtime vs token count
- Extend to other datasets beyond GSM8K


In [ ]:
# Save final results
final_output = {
    "model_name": model_name,
    "config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "n_samples": N_SAMPLES,
        "max_questions": MAX_QUESTIONS,
    },
    "part1_baseline": baseline_metrics,
    "part2_self_consistency": sc_metrics,
    "part3_verifier": verifier_metrics,
    "part5_revisions": {"last": rev_last_metrics, "majority_vote": rev_mv_metrics, "verifier": rev_ver_metrics},
    "part6_beam_search": beam_metrics,
    "part7_policies": {"policy_a": polA_metrics, "policy_b": polB_metrics, "policy_c": polC_metrics},
}

with open("gsm8k_ttc_results.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("✅ Results saved to gsm8k_ttc_results.json")


In [ ]:
# Print total runtime
NOTEBOOK_END_TIME = time.time()
total_runtime = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME

print("\n" + "=" * 80)
print("NOTEBOOK SUMMARY")
print("=" * 80)
print(f"Total runtime: {total_runtime / 60:.2f} minutes ({total_runtime:.0f} seconds)")
print(f"Model: {model_name}")
print(f"Evaluation subset: {len(eval_df)} examples (max {MAX_QUESTIONS})")
print(f"Methods tested: Baseline, Self-Consistency, Verifier, Revisions, Beam Search, Routing Policies")
print("=" * 80)
